# Part 2.2 — Forecast & valuation audit: are the assumptions defensible?

Three questions, for the five companies run so far (NVDA, AMD, INTC, AAPL, GOOGL):

1. **Forecast.** What do the default forecast constants imply for each company, and what should they be instead?
2. **Valuation.** Can the engine's target price be reproduced by hand, and which assumptions does it make without printing them?
3. **Sensitivity.** How much does the target price move with the operating forecast, and how much with the
   valuation constants?

Everything is computed offline from the saved Step 1/Step 2 outputs in `FinRobot/output/`, using the pipeline's own
functions (`calculate_growth_and_forecasts`, `ValuationEngine`, `SensitivityAnalyzer`). The only network calls are
cached in `audit/data/`: shares outstanding (FMP `/stable/shares-float`, the same source Step 2 uses) and analyst
revenue consensus (Yahoo Finance).

Outputs: `audit/forecast_assumptions.csv`, `audit/valuation_assumptions.csv`, `audit/valuation_sensitivity.csv`.

In [1]:
import json
import logging
import re
import sys
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

ROOT = Path.cwd().parent if Path.cwd().name == "audit" else Path.cwd()
SRC = ROOT / "FinRobot" / "finrobot_equity" / "core" / "src"
OUT = ROOT / "FinRobot" / "output"
CACHE = ROOT / "audit" / "data"
CACHE.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC))

from modules.common_utils import load_config, get_api_key
from modules.financial_data_processor import calculate_growth_and_forecasts
from modules.market_data_api import _fmp_get
from modules.sensitivity_analyzer import SensitivityAnalyzer
from modules.valuation_engine import ValuationEngine

logging.getLogger().setLevel(logging.WARNING)  # the engine logs every step at INFO

TICKERS = ["NVDA", "AMD", "INTC", "AAPL", "GOOGL"]
FORECAST_COLS = ["2025E", "2026E", "2027E"]
DEFAULTS = {"growth": [0.05, 0.06, 0.04], "margin_improvement": 0.01, "sga_margin_change": -0.005}
B = 1e9
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 110)


def num(x) -> float:
    s = str(x).rstrip("%").replace(",", "")
    try:
        return float(s)
    except ValueError:
        return np.nan


def actual_cols(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c.endswith("A")]


def restore_types(df: pd.DataFrame) -> pd.DataFrame:
    """Reading the CSV back makes every column text; restore numbers, keep 'x.x%' and 'N/A' as the pipeline has them."""
    def cell(x):
        if isinstance(x, str) and (x.endswith("%") or x == "N/A"):
            return x
        try:
            return float(x)
        except (TypeError, ValueError):
            return x
    out = df.copy()
    for c in out.columns[1:]:
        out[c] = out[c].map(cell).astype(object)
    return out


def load_run(t: str) -> dict:
    a = OUT / t / "analysis"
    log_path = next(p for p in (OUT / t / "step2_full.log", OUT / t / "step2.log") if p.exists())
    log = log_path.read_text(encoding="utf-8", errors="replace")
    synth = re.search(r"Synthesized target: \$([\d.]+) \(range \$([\d.]+)-\$([\d.]+), upside (-?[\d.]+)%\)", log)
    return {
        "analysis": restore_types(pd.read_csv(a / "financial_metrics_and_forecasts.csv")),
        "income": pd.read_csv(a / "income_statement_raw_data.csv"),
        "balance": pd.read_csv(a / "balance_sheet_raw_data.csv"),
        "cash_flow": pd.read_csv(a / "cash_flow_raw_data.csv"),
        "key_metrics": pd.read_csv(a / "key_metrics_raw_data.csv"),
        "peer_ev": pd.read_csv(a / "peer_ev_ebitda_comparison.csv", index_col=0),
        "price": float(re.search(r"Share Price: \$([\d,.]+)", log).group(1).replace(",", "")),
        "report": {
            "EV/EBITDA": float(re.search(r"EV/EBITDA valuation: \$([\d.]+)", log).group(1)),
            "DCF": float(re.search(r"DCF valuation: \$([\d.]+)", log).group(1)),
            "target": float(synth.group(1)), "low": float(synth.group(2)), "high": float(synth.group(3)),
            "upside %": float(synth.group(4)),
        },
    }


RUNS = {t: load_run(t) for t in TICKERS}


def cached(name: str, fetch, keys):
    """JSON cache in audit/data: fetch(key) runs only for keys not cached yet."""
    path = CACHE / name
    data = json.loads(path.read_text()) if path.exists() else {}
    for k in keys:
        if k not in data:
            data[k] = fetch(k)
            path.write_text(json.dumps(data, indent=1))
    return {k: data[k] for k in keys}


_fmp_key = None
def fetch_shares(t):
    global _fmp_key
    _fmp_key = _fmp_key or get_api_key(load_config(str(SRC.parent / "config" / "config.ini")), section="API_KEYS", key="fmp_api_key")
    row = _fmp_get("shares-float", _fmp_key, symbol=t)[0]
    return {"shares": row["outstandingShares"], "as_of": row["date"], "retrieved": str(dt.date.today())}


def fetch_consensus(t):
    est = yf.Ticker(t).revenue_estimate
    out = {p: {c: float(est.loc[p, c]) for c in ("avg", "growth", "numberOfAnalysts", "yearAgoRevenue")} for p in ("0y", "+1y")}
    out["retrieved"] = str(dt.date.today())
    return out


SHARES = {t: v["shares"] for t, v in cached("shares_outstanding.json", fetch_shares, TICKERS).items()}
CONS = cached("consensus_revenue.json", fetch_consensus, TICKERS)
pd.DataFrame({t: {"price": RUNS[t]["price"], "shares (bn)": SHARES[t] / B, "report target": RUNS[t]["report"]["target"]}
              for t in TICKERS}).T

,price,shares (bn),report target
NVDA,238.90,24.221000,82.24
AMD,631.75,1.630600,111.98
INTC,116.19,5.044000,40.24
AAPL,332.89,14.594180,160.33
GOOGL,346.47,12.102111,215.98


## 1. Forecast: what the defaults imply

How Step 1 builds the forecast (`financial_data_processor.calculate_growth_and_forecasts`):

| Mechanic | Code | Comment |
|---|---|---|
| Revenue growth = CLI constants `5% / 6% / 4%`, applied to the latest actual year | `generate_financial_analysis.py:163-176` | Same numbers for every company |
| The three forecast columns are always labelled `2025E, 2026E, 2027E` | `generate_financial_analysis.py:166-168` | They are really FY+1..FY+3 (see 2.1) |
| Gross ("contribution") margin +1pp per year | `financial_data_processor.py:357-363` | |
| SG&A margin −0.5pp per year, cumulative | `financial_data_processor.py:366-376` | |
| **Forecast EBITDA = gross profit − SG&A** | `financial_data_processor.py:414-416` | **R&D and other opex are dropped**, while actual EBITDA is FMP's full EBITDA |
| The `EBITDA Margin` improvement setting is overwritten by the line above; `ebitda_growth_factor=1.05` is never read | `financial_data_processor.py:357-363`, `:414-420` | Dead configuration |
| EPS grows by `(1+g) × 1.01` a year | `financial_data_processor.py:383-390` | The 1pp margin improvement is applied as 1% EPS growth |
| P/E falls 5% a year | `financial_data_processor.py:393-399` | Assumed multiple compression |

The table compares each company's history and the analyst consensus (Yahoo Finance; `0y` = the fiscal year after
the latest actual, confirmed by `yearAgoRevenue` matching the latest reported revenue) with what the defaults produce.

In [2]:
def default_forecast_check(t):
    """Re-run the pipeline's forecast with the default constants; it must reproduce the saved CSV."""
    saved = RUNS[t]["analysis"]
    redo = forecast(t, DEFAULTS["growth"], DEFAULTS["margin_improvement"], DEFAULTS["sga_margin_change"])
    a, b = saved.set_index("metrics")[FORECAST_COLS].map(num), redo.set_index("metrics")[FORECAST_COLS].map(num)
    return bool(np.allclose(a.loc[b.index], b, equal_nan=True, rtol=1e-6))


def forecast(t, growth, margin_improvement, sga_change):
    fm = RUNS[t]["analysis"]
    hist = fm[["metrics"] + actual_cols(fm)].copy()
    cfg = {
        "revenue_base_year": actual_cols(fm)[-1],
        "revenue_growth_assumptions": dict(zip(FORECAST_COLS, growth)),
        "ebitda_growth_factor": 1.05,
        "margin_improvement": {"Contribution Margin": margin_improvement, "EBITDA Margin": margin_improvement},
        "sga_margin_change": sga_change,
    }
    return calculate_growth_and_forecasts(hist, cfg)


rows = []
for t in TICKERS:
    fm = RUNS[t]["analysis"].set_index("metrics")
    acts = actual_cols(fm)
    last = acts[-1]
    rev = [num(fm.loc["Revenue", c]) for c in acts]
    inc = RUNS[t]["income"]
    rnd_share = inc["researchAndDevelopmentExpenses"].iloc[0] / inc["revenue"].iloc[0]
    c = CONS[t]
    rows.append({
        "ticker": t,
        "latest actual": last,
        "growth latest FY": rev[-1] / rev[-2] - 1,
        "3y revenue CAGR": (rev[-1] / rev[-4]) ** (1 / 3) - 1,
        "consensus FY+1 growth": c["0y"]["growth"],
        "consensus FY+2 growth": c["+1y"]["growth"],
        "default FY+1 growth": DEFAULTS["growth"][0],
        "FY+1 revenue: pipeline (bn)": num(fm.loc["Revenue", "2025E"]) / B,
        "FY+1 revenue: consensus (bn)": c["0y"]["avg"] / B,
        "pipeline vs consensus FY+1": num(fm.loc["Revenue", "2025E"]) / c["0y"]["avg"] - 1,
        "consensus base = latest actual": abs(c["0y"]["yearAgoRevenue"] / rev[-1] - 1) < 0.001,
        "EBITDA margin latest actual": num(fm.loc["EBITDA Margin", last]) / 100,
        "EBITDA margin FY+1 (pipeline)": num(fm.loc["EBITDA Margin", "2025E"]) / 100,
        "R&D % of revenue (dropped from forecast EBITDA)": rnd_share,
        "saved forecast reproduced": default_forecast_check(t),
    })
implied = pd.DataFrame(rows).set_index("ticker")
implied.T

ticker,NVDA,AMD,INTC,AAPL,GOOGL
latest actual,2026A,2025A,2025A,2025A,2025A
growth latest FY,0.654735,0.343378,-0.00467,0.064255,0.150901
3y revenue CAGR,1.000451,0.136436,-0.057129,0.018125,0.125117
consensus FY+1 growth,0.9064,0.4708,0.1934,0.1482,0.2378
consensus FY+2 growth,0.67,0.7349,0.1419,0.1049,0.2334
default FY+1 growth,0.05,0.05,0.05,0.05,0.05
FY+1 revenue: pipeline (bn),226.7349,36.37095,55.49565,436.96905,422.9778
FY+1 revenue: consensus (bn),411.65341,50.948758,63.076963,477.854312,498.628983
pipeline vs consensus FY+1,-0.449209,-0.286127,-0.120191,-0.08556,-0.151718
consensus base = latest actual,True,True,True,True,True


### What I changed the assumptions to, and why

* **Revenue, FY+1 and FY+2: analyst consensus** (Yahoo Finance, 25–60 analysts per company). It is the market's
  own expectation, it differs by company, and it is observable, so the forecast stops being a guess applied equally
  to a hypergrowth chip designer and a mature hardware company.
* **Revenue, FY+3: consensus FY+2 growth faded halfway towards 5%.** There is no consensus beyond FY+2; fading
  towards a mature growth rate is a stated judgment, not a fitted number.
* **Margins: held at the latest actual level** (`--margin-improvement 0 --sga-margin-improvement 0`). Nothing in the
  data supports a mechanical +1pp gross margin and −0.5pp SG&A every year for every company.

These are CLI arguments, so the revised forecast can be produced by the unmodified pipeline. The EBITDA formula
(R&D dropped) is code, not configuration, and stays wrong under any assumptions; fixing it belongs in Part 3.

In [3]:
LONG_RUN_GROWTH = 0.05


def revised_growth(t):
    g1, g2 = CONS[t]["0y"]["growth"], CONS[t]["+1y"]["growth"]
    return [g1, g2, (g2 + LONG_RUN_GROWTH) / 2]


rows = []
for t in TICKERS:
    g = revised_growth(t)
    base = forecast(t, DEFAULTS["growth"], DEFAULTS["margin_improvement"], DEFAULTS["sga_margin_change"]).set_index("metrics")
    new = forecast(t, g, 0.0, 0.0).set_index("metrics")
    rows.append({
        "ticker": t,
        "default growth FY+1/2/3": "5.0% / 6.0% / 4.0%",
        "revised growth FY+1/2/3": " / ".join(f"{x:.1%}" for x in g),
        "FY+3 revenue default (bn)": num(base.loc["Revenue", "2027E"]) / B,
        "FY+3 revenue revised (bn)": num(new.loc["Revenue", "2027E"]) / B,
        "FY+3 EBITDA default (bn)": num(base.loc["EBITDA", "2027E"]) / B,
        "FY+3 EBITDA revised (bn)": num(new.loc["EBITDA", "2027E"]) / B,
        "CLI flags for the revised run": (f"--revenue-growth-2025 {g[0]:.4f} --revenue-growth-2026 {g[1]:.4f} "
                                          f"--revenue-growth-2027 {g[2]:.4f} --margin-improvement 0 --sga-margin-improvement 0"),
    })
assumptions = pd.DataFrame(rows).set_index("ticker")
assumptions.join(implied[["growth latest FY", "3y revenue CAGR"]]).to_csv(ROOT / "audit" / "forecast_assumptions.csv")
assumptions.round(1)

,default growth FY+1/2/3,revised growth FY+1/2/3,FY+3 revenue default (bn),FY+3 revenue revised (bn),FY+3 EBITDA default (bn),FY+3 EBITDA revised (bn),CLI flags for the revised run
ticker,,,,,,,
NVDA,5.0% / 6.0% / 4.0%,90.6% / 67.0% / 36.0%,250.0,935.0,183.7,645.1,--revenue-growth-2025 0.9064 --revenue-growth-2026 0.6700 --revenue-growth-2027 0.3600 --margin-improvemen...
AMD,5.0% / 6.0% / 4.0%,47.1% / 73.5% / 39.2%,40.1,123.1,16.8,46.2,--revenue-growth-2025 0.4708 --revenue-growth-2026 0.7349 --revenue-growth-2027 0.3925 --margin-improvemen...
INTC,5.0% / 6.0% / 4.0%,19.3% / 14.2% / 9.6%,61.2,78.9,18.7,20.6,--revenue-growth-2025 0.1934 --revenue-growth-2026 0.1419 --revenue-growth-2027 0.0960 --margin-improvemen...
AAPL,5.0% / 6.0% / 4.0%,14.8% / 10.5% / 7.7%,481.7,568.9,215.8,229.2,--revenue-growth-2025 0.1482 --revenue-growth-2026 0.1049 --revenue-growth-2027 0.0774 --margin-improvemen...
GOOGL,5.0% / 6.0% / 4.0%,23.8% / 23.3% / 14.2%,466.3,702.2,241.1,331.4,--revenue-growth-2025 0.2378 --revenue-growth-2026 0.2334 --revenue-growth-2027 0.1417 --margin-improvemen...


## 2. Valuation: reproducing NVIDIA's $82.24 by hand

`create_equity_report.py:752-776` passes the engine three things: the forecast table, the share price and the share
count. It then runs EV/EBITDA, DCF and peer comparison and takes a confidence-weighted average. Below, every step is
first computed with plain arithmetic, using only the constants found in `valuation_engine.py`, and then compared with a
fresh engine run and with the numbers in the Step 2 log.

In [4]:
T = "NVDA"
r = RUNS[T]
fm = r["analysis"].set_index("metrics")
last_col = [c for c in r["analysis"].columns if c.endswith(("A", "E"))][-1]   # _get_metric(): latest column
ebitda = num(fm.loc["EBITDA", last_col])
shares, price = SHARES[T], r["price"]

# EV/EBITDA (valuation_engine.py:70-131); key_metrics is not passed, so the multiple falls back to 12.0 +/- 3.0
mult, mult_sd = 12.0, 3.0
ev = ebitda * mult
nd_ev = 0.10 * ev                                     # "net debt = 10% of EV"
ev_tp = (ev - nd_ev) / shares
ev_lo = (ebitda * (mult - mult_sd) - nd_ev) / shares
ev_hi = (ebitda * (mult + mult_sd) - nd_ev) / shares

# DCF (valuation_engine.py:203-282); no free_cash_flow passed, so FCF = 60% of EBITDA
wacc, tg, g15, g610, n = 0.10, 0.025, 0.10, 0.05, 10
fcf = 0.6 * ebitda
pv, cf = 0.0, fcf
for year in range(1, n + 1):
    cf *= 1 + (g15 if year <= 5 else g610)
    pv += cf / (1 + wacc) ** year
tv = cf * (1 + tg) / (wacc - tg)
dcf_ev = pv + tv / (1 + wacc) ** n
nd_dcf = 0.10 * dcf_ev
dcf_tp = (dcf_ev - nd_dcf) / shares
dcf_lo = (pv + tv / (1 + wacc + 0.01) ** n - nd_dcf) / shares    # only the terminal value is re-discounted
dcf_hi = (pv + tv / (1 + wacc - 0.01) ** n - nd_dcf) / shares

# Synthesis (valuation_engine.py:312-358): weights are the methods' 'confidence' values, normalised
target = (ev_tp * 0.7 + dcf_tp * 0.5) / (0.7 + 0.5)
hand = {"EV/EBITDA": ev_tp, "DCF": dcf_tp, "target": target, "low": min(ev_lo, dcf_lo),
        "high": max(ev_hi, dcf_hi), "upside %": (target / price - 1) * 100}

steps = pd.DataFrame([
    ("EBITDA input", f"'{last_col}' column of the forecast", ebitda / B, "USD bn"),
    ("Shares outstanding", "FMP /shares-float", shares / B, "bn"),
    ("EV/EBITDA: EV", f"EBITDA x {mult:.0f}", ev / B, "USD bn"),
    ("EV/EBITDA: net debt", "10% of EV", nd_ev / B, "USD bn"),
    ("EV/EBITDA: price", "(EV - net debt) / shares", ev_tp, "USD"),
    ("DCF: FCF year 0", "60% of EBITDA", fcf / B, "USD bn"),
    ("DCF: PV of FCF yrs 1-10", "10% growth yrs 1-5, 5% yrs 6-10, WACC 10%", pv / B, "USD bn"),
    ("DCF: terminal value", "FCF10 x 1.025 / (10% - 2.5%)", tv / B, "USD bn"),
    ("DCF: EV", "PV FCF + PV terminal", dcf_ev / B, "USD bn"),
    ("DCF: price", "(EV - 10% of EV) / shares", dcf_tp, "USD"),
    ("Target", "(0.7 x EV/EBITDA + 0.5 x DCF) / 1.2", target, "USD"),
], columns=["step", "formula", "value", "unit"])
steps.round(2)

,step,formula,value,unit
0,EBITDA input,'2027E' column of the forecast,183.72,USD bn
1,Shares outstanding,FMP /shares-float,24.22,bn
2,EV/EBITDA: EV,EBITDA x 12,2204.58,USD bn
3,EV/EBITDA: net debt,10% of EV,220.46,USD bn
4,EV/EBITDA: price,(EV - net debt) / shares,81.92,USD
5,DCF: FCF year 0,60% of EBITDA,110.23,USD bn
6,DCF: PV of FCF yrs 1-10,"10% growth yrs 1-5, 5% yrs 6-10, WACC 10%",1031.54,USD bn
7,DCF: terminal value,FCF10 x 1.025 / (10% - 2.5%),3096.48,USD bn
8,DCF: EV,PV FCF + PV terminal,2225.37,USD bn
9,DCF: price,(EV - 10% of EV) / shares,82.69,USD


In [5]:
def run_engine(t, analysis=None, multiple=None, dcf=None):
    """The engine exactly as create_equity_report.py:752-776 runs it."""
    run = RUNS[t]
    eng = ValuationEngine({"analysis": run["analysis"] if analysis is None else analysis,
                           "current_price": run["price"], "shares_outstanding": SHARES[t]},
                          {"ev_ebitda": run["peer_ev"]})
    eng.calculate_ev_ebitda_valuation(multiple)
    eng.calculate_dcf_valuation(dcf)
    peer = eng.calculate_peer_comparison_valuation()
    return eng, eng.synthesize_valuation(), peer


eng, syn, peer = run_engine(T)
engine = {"EV/EBITDA": eng.valuation_results[0].target_price, "DCF": eng.valuation_results[1].target_price,
          "target": syn["target_price"], "low": syn["range"][0], "high": syn["range"][1], "upside %": syn["upside"]}
reconcile = pd.DataFrame({"by hand": hand, "engine (re-run)": engine, "Step 2 log": r["report"]})
reconcile["hand - log"] = reconcile["by hand"] - reconcile["Step 2 log"]
print(f"methods used by the engine: {[m.method for m in eng.valuation_results]}; peer comparison target: {peer.target_price}")
reconcile.round(2)

methods used by the engine: ['EV/EBITDA', 'DCF']; peer comparison target: 0


,by hand,engine (re-run),Step 2 log,hand - log
EV/EBITDA,81.92,81.92,81.92,-0.00
DCF,82.69,82.69,82.69,-0.00
target,82.24,82.24,82.24,-0.00
low,59.16,59.16,59.16,0.00
high,104.67,104.67,104.67,0.00
upside %,-65.58,-65.58,-65.60,0.02


**Why peer comparison never contributes.** `create_equity_report.py` passes `{"ev_ebitda": <DataFrame of all
peers>}`, but the engine iterates `peer_data.items()` expecting `{ticker: {"ev_ebitda": multiple}}`
(`valuation_engine.py:153-156`). The only "ticker" it finds is the string `"ev_ebitda"`, which has no
`"ev_ebitda"` key, so no multiples are collected and the method returns 0 and is dropped without a warning. Passing the
shape the engine expects shows what the third method would have said:

In [6]:
peer_ev = r["peer_ev"]
expected_shape = {tk: {"ev_ebitda": peer_ev[tk].dropna().iloc[-1]} for tk in peer_ev.columns if tk != T}
fixed = ValuationEngine({"analysis": r["analysis"], "current_price": price, "shares_outstanding": shares}, expected_shape)
res = fixed.calculate_peer_comparison_valuation()
print("peer multiples used:", {k: round(v["ev_ebitda"], 1) for k, v in expected_shape.items()})
print(f"peer-comparison target with the expected input shape: ${res.target_price:.2f} "
      f"(range ${res.low_estimate:.2f}-${res.high_estimate:.2f})")

peer multiples used: {'AMD': 47.8, 'INTC': 14.5}
peer-comparison target with the expected input shape: $212.79 (range $86.32-$339.26)


### Assumption register

Every assumption behind the engine's number, and whether a reader of the report can see it. The values are NVIDIA's
run; the constants are the same for every company.

In [7]:
register = pd.DataFrame([
    ("EBITDA input", f"'{last_col}' column = FY+3 forecast, ${ebitda/B:.1f}bn, built as gross profit - SG&A (R&D excluded)",
     "valuation_engine.py:58-60; financial_data_processor.py:414-416", "Number printed, year and definition not"),
    ("EV/EBITDA target multiple", "12.0x (default: key_metrics is never passed to the engine)",
     "valuation_engine.py:88-98; create_equity_report.py:755-760", "Printed, but labelled 'Historical Avg Multiple'"),
    ("EV/EBITDA range", "target multiple +/- 3.0x", "valuation_engine.py:95, 110-111", "No"),
    ("Net debt", "10% of enterprise value, in both methods (balance sheet ignored)", "valuation_engine.py:104, 256", "No"),
    ("Shares outstanding", f"{shares/B:.2f}bn from FMP /shares-float", "create_equity_report.py:758", "No"),
    ("Free cash flow", "60% of the EBITDA input (no FCF is passed)", "valuation_engine.py:228-231", "No"),
    ("FCF growth", "10% in years 1-5, 5% in years 6-10", "valuation_engine.py:217-218", "Yes"),
    ("Discount rate (WACC)", "10%, same for every company", "valuation_engine.py:220", "No (not in the assumptions column)"),
    ("Terminal growth", "2.5%", "valuation_engine.py:219", "Yes"),
    ("Projection horizon", "10 years", "valuation_engine.py:221", "No"),
    ("DCF timing", "the FY+3 forecast is treated as the year-0 cash flow and grown/discounted from today",
     "valuation_engine.py:228-245", "No"),
    ("DCF range", "WACC +/- 1% applied to discounting the terminal value only (terminal value itself unchanged)",
     "valuation_engine.py:263-266", "No"),
    ("Method weights", "'confidence' 0.7 (EV/EBITDA), 0.5 (DCF), 0.6 (peers), normalised",
     "valuation_engine.py:125, 276, 195, 327-333", "Printed as 70% / 50% (sums to 120%)"),
    ("Peer comparison", "always dropped: peer data passed in the wrong shape",
     "create_equity_report.py:764-766; valuation_engine.py:153-165", "No"),
    ("Target range", "lowest method low to highest method high", "valuation_engine.py:335-340", "Yes"),
    ("Page-1 rating", "from FMP's analyst consensus target vs price (>=15% upside = Buy), not from this valuation",
     "html_template_professional.py:961-990", "No"),
], columns=["assumption", "value / rule", "code", "shown in report?"])
register.to_csv(ROOT / "audit" / "valuation_assumptions.csv", index=False)
register

,assumption,value / rule,code,shown in report?
0,EBITDA input,"'2027E' column = FY+3 forecast, $183.7bn, built as gross profit - SG&A (R&D excluded)",valuation_engine.py:58-60; financial_data_processor.py:414-416,"Number printed, year and definition not"
1,EV/EBITDA target multiple,12.0x (default: key_metrics is never passed to the engine),valuation_engine.py:88-98; create_equity_report.py:755-760,"Printed, but labelled 'Historical Avg Multiple'"
2,EV/EBITDA range,target multiple +/- 3.0x,"valuation_engine.py:95, 110-111",No
3,Net debt,"10% of enterprise value, in both methods (balance sheet ignored)","valuation_engine.py:104, 256",No
4,Shares outstanding,24.22bn from FMP /shares-float,create_equity_report.py:758,No
5,Free cash flow,60% of the EBITDA input (no FCF is passed),valuation_engine.py:228-231,No
6,FCF growth,"10% in years 1-5, 5% in years 6-10",valuation_engine.py:217-218,Yes
7,Discount rate (WACC),"10%, same for every company",valuation_engine.py:220,No (not in the assumptions column)
8,Terminal growth,2.5%,valuation_engine.py:219,Yes
9,Projection horizon,10 years,valuation_engine.py:221,No


### What the undisclosed constants are worth

Same engine formulas, replacing one constant at a time (cumulatively) with the company's own data. This is not my
valuation of NVIDIA; it shows which inputs actually decide the answer.

In [8]:
def engine_formula(ebitda, multiple, fcf, shares, net_debt=None, wacc=0.10, tg=0.025, g15=0.10, g610=0.05, n=10):
    """The engine's EV/EBITDA + DCF + 0.7/0.5 weighting; net_debt=None means its '10% of EV' rule."""
    ev = ebitda * multiple
    p1 = (ev - (0.1 * ev if net_debt is None else net_debt)) / shares
    pv, cf = 0.0, fcf
    for year in range(1, n + 1):
        cf *= 1 + (g15 if year <= 5 else g610)
        pv += cf / (1 + wacc) ** year
    dcf_ev = pv + cf * (1 + tg) / (wacc - tg) / (1 + wacc) ** n
    p2 = (dcf_ev - (0.1 * dcf_ev if net_debt is None else net_debt)) / shares
    return (0.7 * p1 + 0.5 * p2) / 1.2


def bridge(t):
    run, fm_t = RUNS[t], RUNS[t]["analysis"].set_index("metrics")
    last_f = [c for c in run["analysis"].columns if c.endswith(("A", "E"))][-1]
    last_a = actual_cols(run["analysis"])[-1]
    e_fc, e_act = num(fm_t.loc["EBITDA", last_f]), num(fm_t.loc["EBITDA", last_a])
    bal, cfs, km = run["balance"].iloc[0], run["cash_flow"].iloc[0], run["key_metrics"]
    net_debt = bal["totalDebt"] - bal["cashAndCashEquivalents"]
    hist_mult = km["evToEBITDA"].mean()
    sh, px = SHARES[t], run["price"]
    steps = [
        ("engine as shipped", engine_formula(e_fc, 12, 0.6 * e_fc, sh)),
        ("+ actual net debt", engine_formula(e_fc, 12, 0.6 * e_fc, sh, net_debt)),
        (f"+ latest actual EBITDA ({last_a}) instead of FY+3 forecast", engine_formula(e_act, 12, 0.6 * e_act, sh, net_debt)),
        (f"+ own {len(km)}y average EV/EBITDA ({hist_mult:.1f}x) instead of 12x", engine_formula(e_act, hist_mult, 0.6 * e_act, sh, net_debt)),
        ("+ actual free cash flow instead of 60% of EBITDA", engine_formula(e_act, hist_mult, cfs["freeCashFlow"], sh, net_debt)),
    ]
    return pd.DataFrame([{"ticker": t, "step": s, "target": v, "vs price": v / px - 1} for s, v in steps])


assert abs(bridge(T)["target"].iloc[0] - hand["target"]) < 0.01   # step 0 reproduces the engine
bridges = pd.concat([bridge(t) for t in TICKERS], ignore_index=True)
bridges.to_csv(ROOT / "audit" / "valuation_bridge.csv", index=False)
bridges.style.format({"target": "${:.2f}", "vs price": "{:+.0%}"}).hide(axis="index")

ticker,step,target,vs price
NVDA,engine as shipped,$82.24,-66%
NVDA,+ actual net debt,$91.34,-62%
NVDA,+ latest actual EBITDA (2026A) instead of FY+3 forecast,$71.86,-70%
NVDA,+ own 5y average EV/EBITDA (48.9x) instead of 12x,$200.27,-16%
NVDA,+ actual free cash flow instead of 60% of EBITDA,$203.73,-15%
AMD,engine as shipped,$111.98,-82%
AMD,+ actual net debt,$125.07,-80%
AMD,+ latest actual EBITDA (2025A) instead of FY+3 forecast,$54.40,-91%
AMD,+ own 5y average EV/EBITDA (40.7x) instead of 12x,$128.97,-80%
AMD,+ actual free cash flow instead of 60% of EBITDA,$141.19,-78%


## 3. Sensitivity: what moves the target price?

### The shipped `SensitivityAnalyzer`

Step 1's sensitivity analysis (`sensitivity_analyzer.py`) never touches the target price. Its "revenue growth ±5%"
scales the forecast revenue **level** by ±5%, its margin grid shifts the EBITDA margin ±2pp, and its
"95% confidence interval" is `base ± 1.96 × 15% × base` with the 15% hard-coded (`sensitivity_analyzer.py:240`),
so every company gets the same ±29.4% band regardless of its history.

In [9]:
rows = []
for t in TICKERS:
    sa = SensitivityAnalyzer(RUNS[t]["analysis"])
    lo, hi = sa.calculate_confidence_interval("Revenue")
    base = sa.confidence_intervals["Revenue"]["base"]
    rows.append({"ticker": t, "revenue FY+3 (bn)": base / B, "'95% CI' low": lo / B, "high": hi / B,
                 "half-width / base": (hi - base) / base})
print(SensitivityAnalyzer(RUNS[T]["analysis"]).generate_sensitivity_table().to_string())
pd.DataFrame(rows).set_index("ticker").round(3)

               Margin -2.0% Margin -1.0% Margin +0.0% Margin +1.0% Margin +2.0%
Revenue Growth                                                                 
-5.0%               $169.8B      $172.2B      $174.5B      $176.9B      $179.3B
-2.5%               $174.2B      $176.7B      $179.1B      $181.6B      $184.0B
+0.0%               $178.7B      $181.2B      $183.7B      $186.2B      $188.7B
+2.5%               $183.2B      $185.7B      $188.3B      $190.9B      $193.4B
+5.0%               $187.7B      $190.3B      $192.9B      $195.5B      $198.1B


,revenue FY+3 (bn),'95% CI' low,high,half-width / base
ticker,,,,
NVDA,249.953,176.467,323.439,0.294
AMD,40.095,28.307,51.883,0.294
INTC,61.178,43.192,79.165,0.294
AAPL,481.715,340.091,623.339,0.294
GOOGL,466.291,329.201,603.380,0.294


### Target price vs revenue and margin (NVDA)

Since the engine only reads the last forecast column's EBITDA, a revenue shock `dr` and a margin shock `dm` (in pp)
change its input to `revenue × (1+dr) × (margin + dm)`. Each cell is a full engine run; values are % change vs the
base target.

In [10]:
def shocked(t, dr=0.0, dm=0.0):
    a = RUNS[t]["analysis"].copy()
    last_f = [c for c in a.columns if c.endswith(("A", "E"))][-1]
    i_rev, i_e = a.index[a["metrics"] == "Revenue"][0], a.index[a["metrics"] == "EBITDA"][0]
    rev, e = num(a.at[i_rev, last_f]), num(a.at[i_e, last_f])
    margin = e / rev
    a[last_f] = a[last_f].astype(object)
    a.at[i_rev, last_f] = rev * (1 + dr)
    a.at[i_e, last_f] = rev * (1 + dr) * (margin + dm / 100)
    return run_engine(t, a)[1]["target_price"]


base_tp = shocked(T)
grid = pd.DataFrame({f"margin {dm:+.0f}pp": {f"revenue {dr:+.1%}": shocked(T, dr, dm) / base_tp - 1
                                               for dr in (-0.05, -0.025, 0, 0.025, 0.05)}
                     for dm in (-2, -1, 0, 1, 2)})
print(f"NVDA base engine target ${base_tp:.2f}")
grid.map("{:+.1%}".format)

NVDA base engine target $82.24


,margin -2pp,margin -1pp,margin +0pp,margin +1pp,margin +2pp
revenue -5.0%,-7.6%,-6.3%,-5.0%,-3.7%,-2.4%
revenue -2.5%,-5.2%,-3.8%,-2.5%,-1.2%,+0.2%
revenue +0.0%,-2.7%,-1.4%,+0.0%,+1.4%,+2.7%
revenue +2.5%,-0.3%,+1.1%,+2.5%,+3.9%,+5.3%
revenue +5.0%,+2.1%,+3.6%,+5.0%,+6.4%,+7.9%


### All five companies: operating forecast vs valuation constants

For each company, how far the target moves when the **operating forecast** changes (±5% revenue, ±2pp margin,
±5pp growth in every forecast year, the consensus-based forecast from section 1) versus when one **valuation
constant** changes (WACC, terminal growth, the 12x multiple).

In [11]:
rows = []
for t in TICKERS:
    base = run_engine(t)[1]["target_price"]
    hist_mult = RUNS[t]["key_metrics"]["evToEBITDA"].mean()
    pct = lambda v: v / base - 1

    def with_growth(delta=0.0, revised=False):
        g = revised_growth(t) if revised else [x + delta for x in DEFAULTS["growth"]]
        mi, sg = (0.0, 0.0) if revised else (DEFAULTS["margin_improvement"], DEFAULTS["sga_margin_change"])
        f = forecast(t, g, mi, sg)
        f = f.drop(columns=[c for c in f.columns if c == "CAGR"])
        return run_engine(t, f)[1]["target_price"]

    rows.append({
        "ticker": t, "price": RUNS[t]["price"], "engine target": base,
        "revenue -5%": pct(shocked(t, -0.05)), "revenue +5%": pct(shocked(t, 0.05)),
        "margin -2pp": pct(shocked(t, 0, -2)), "margin +2pp": pct(shocked(t, 0, 2)),
        "growth -5pp/yr": pct(with_growth(-0.05)), "growth +5pp/yr": pct(with_growth(0.05)),
        "consensus forecast": pct(with_growth(revised=True)),
        "WACC 8%": pct(run_engine(t, dcf={"wacc": 0.08})[1]["target_price"]),
        "WACC 12%": pct(run_engine(t, dcf={"wacc": 0.12})[1]["target_price"]),
        "terminal g 1.5%": pct(run_engine(t, dcf={"terminal_growth": 0.015})[1]["target_price"]),
        "terminal g 3.5%": pct(run_engine(t, dcf={"terminal_growth": 0.035})[1]["target_price"]),
        f"own avg multiple": pct(run_engine(t, multiple=hist_mult)[1]["target_price"]),
        "own avg multiple (x)": hist_mult,
    })
sens = pd.DataFrame(rows).set_index("ticker")
sens.to_csv(ROOT / "audit" / "valuation_sensitivity.csv")
fmt = {c: "{:+.0%}".format for c in sens.columns if c not in ("price", "engine target", "own avg multiple (x)")}
sens.style.format(fmt | {"price": "${:.2f}", "engine target": "${:.2f}", "own avg multiple (x)": "{:.1f}x"})

,price,engine target,revenue -5%,revenue +5%,margin -2pp,margin +2pp,growth -5pp/yr,growth +5pp/yr,consensus forecast,WACC 8%,WACC 12%,terminal g 1.5%,terminal g 3.5%,own avg multiple,own avg multiple (x)
ticker,,,,,,,,,,,,,,,
NVDA,$238.90,$82.24,-5%,+5%,-3%,+3%,-14%,+15%,+251%,+16%,-9%,-3%,+4%,+179%,48.9x
AMD,$631.75,$111.98,-5%,+5%,-5%,+5%,-14%,+15%,+174%,+16%,-9%,-3%,+4%,+139%,40.7x
INTC,$116.19,$40.24,-5%,+5%,-7%,+7%,-14%,+15%,+10%,+16%,-9%,-3%,+4%,+95%,31.5x
AAPL,$332.89,$160.33,-5%,+5%,-4%,+4%,-14%,+15%,+6%,+16%,-9%,-3%,+4%,+53%,23.0x
GOOGL,$346.47,$215.98,-5%,+5%,-4%,+4%,-14%,+15%,+37%,+16%,-9%,-3%,+4%,+28%,17.8x


### Why the constants move every company by the same percentage

Both methods are linear in the one EBITDA number the engine reads, and every other input is a constant. The whole
engine therefore reduces to a single multiple applied to FY+3 forecast EBITDA per share, and that multiple is the same
for every company:

In [12]:
k = {}
for t in TICKERS:
    a = RUNS[t]["analysis"].set_index("metrics")
    last_f = [c for c in RUNS[t]["analysis"].columns if c.endswith(("A", "E"))][-1]
    k[t] = RUNS[t]["report"]["target"] * SHARES[t] / num(a.loc["EBITDA", last_f])
closed_form = (0.7 * 12 * 0.9 + 0.5 * (pv + tv / 1.1 ** 10) * 0.9 / ebitda) / 1.2   # from the NVDA hand calculation
print(f"closed form: target = {closed_form:.3f} x FY+3 EBITDA / shares")
pd.Series(k, name="report target x shares / FY+3 EBITDA").round(3)

closed form: target = 10.842 x FY+3 EBITDA / shares


NVDA     10.843
AMD      10.843
INTC     10.842
AAPL     10.842
GOOGL    10.842
Name: report target x shares / FY+3 EBITDA, dtype: float64

## Findings

**Forecast**

1. **The default growth constants are below consensus for every company, by a lot.** FY+1 revenue comes out
   9% (AAPL) to 45% (NVDA) below the analyst consensus. NVIDIA grew 65.5% last year and consensus expects +91%;
   the pipeline assumes +5%.
2. **The forecast formula biases margins the other way.** Forecast EBITDA is gross profit − SG&A, so R&D (8% of
   revenue at Apple, 23–26% at AMD and Intel) disappears in the first forecast year. AMD's EBITDA margin jumps from
   21.0% (actual) to 39.0% (FY+1), Apple's from 34.7% to 41.8%. The forecast is therefore inconsistent with the
   history it extends: revenue too low, margins too high.
3. **What I changed them to:** consensus growth for FY+1 and FY+2, faded halfway to 5% for FY+3, margins held flat.
   These are CLI flags (table above), so the unmodified pipeline can produce the revised forecast. The R&D omission is
   in the code and survives any choice of assumptions.

**Valuation**

4. **The engine's $82.24 for NVIDIA reproduces to the cent by hand**, including the $59.16–$104.67 range.
5. **Ten of the sixteen assumptions are invisible in the report, and three more are shown misleadingly.** Invisible:
   net debt (10% of EV), shares, FCF (60% of EBITDA), WACC, horizon, DCF timing, DCF range rule, multiple range,
   the dropped peer method, and the source of the page-1 rating. Misleading: the 12x default printed as
   "Historical Avg Multiple", method weights printed as 70% + 50%, and an EBITDA figure printed without its year
   or definition.
6. **The two methods are one number.** Both are linear in a single input, so for every company
   `target = 10.84 × FY+3 EBITDA / shares`. The "DCF" adds no information beyond a fixed multiple, and the third
   method (peer comparison) is silently dropped by an input-shape bug; with the expected input it would have said
   $212.79 for NVIDIA.

**Sensitivity**

7. **By the assignment's test the point estimate is not "theater": ±5% revenue moves it exactly ±5%, and ±2pp of
   margin moves it 3–7%.** The reason is that the method is linear, not that the estimate is robust. What actually
   decides the number are choices the reader never sees. Swapping the default growth for consensus moves the target
   by +6% (AAPL) to +251% (NVDA). Using each company's own historical EV/EBITDA instead of 12x moves it +28% to +179%.
   WACC 8%/12% moves it +16%/−9% for every company alike.
8. **The shipped sensitivity analysis never tests the target price**, and its "95% confidence interval" is a fixed
   ±29.4% band (hard-coded 15% volatility) that is identical for all five companies.
9. **The "implied downside" of 38–82% is a property of the method, not a view on the stocks.** Even with each
   company's own net debt, actual EBITDA, historical multiple and actual FCF, the engine stays 15–78% below the
   market price, because a fixed 10%/5% growth DCF cannot express the growth the market is pricing in. Meanwhile
   page 1 shows a Buy/Hold rating that comes from the analyst consensus target, not from this valuation, so the report
   contradicts itself.
